# 05 — Dask integration

`edr-xarray` is fully compatible with Dask. Passing `chunks={...}` to
`xr.open_dataset` wraps each variable in a `dask.array`, and standard
xarray reductions (`.mean`, `.sum`, etc.) execute via Dask's
distributed scheduler.

The library also publishes a `preferred_chunks={"t": 1}` encoding hint,
so even `chunks="auto"` will chunk one timestep per task.

This notebook is gated on `dask` being importable — if it is not
installed, the notebook will print a message and skip the demos.

## Check Dask availability

In [ ]:
try:
    import dask  # noqa: F401
    HAS_DASK = True
except ImportError:
    HAS_DASK = False

print(f"dask available: {HAS_DASK}")
if not HAS_DASK:
    print("install with: uv pip install dask")

In [ ]:
%pip install -q -e ..

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
collection_id = collections[0]["id"]  # or pick any id from the list
collection_url = f"{server}/collections/{collection_id}"

## 1. Open with explicit chunks

Pass `chunks={"t": 1}` to chunk one timestep per task. Each chunk is
a separate cube fetch, computed lazily.

In [ ]:
if HAS_DASK:
    ds = xr.open_dataset(
        collection_url,
        engine="edr",
        chunks={"t": 1},
    )
    da = ds["temperature"]
    print("data type:    ", type(da.data).__name__)  # dask.array.Array
    print("chunks:       ", da.chunks)
    print("nbytes (lazy):", da.data.nbytes)
    ds.close()
else:
    print("skipped — dask not installed")

## 2. Reductions trigger one fetch per chunk

`.mean(dim="t")` builds a Dask graph; calling `.compute()` schedules
the actual cube fetches.

In [ ]:
if HAS_DASK:
    ds = xr.open_dataset(
        collection_url,
        engine="edr",
        chunks={"t": 1},
    )
    mean_t = ds["temperature"].mean(dim="t")
    print("mean_t (lazy):", type(mean_t.data).__name__)
    result = mean_t.compute()
    print("mean_t shape:", result.shape)
    print("mean_t dtype:", result.dtype)
    print(result.values)
    ds.close()
else:
    print("skipped — dask not installed")

## 3. `chunks="auto"` honors `preferred_chunks`

`edr-xarray` declares `preferred_chunks={"t": 1}` in each variable's
encoding. Passing `chunks={}` (or `chunks="auto"`) makes Dask pick the
preferred chunking automatically.

In [ ]:
if HAS_DASK:
    ds = xr.open_dataset(
        collection_url,
        engine="edr",
        chunks={},
    )
    print("chunks (auto):", ds["temperature"].chunks)
    ds.close()
else:
    print("skipped — dask not installed")

## 4. Pickle round-trip

For Dask's multiprocessing or distributed scheduler, the lazy backend
array must be pickleable. `EdrDataStore.__getstate__` drops the
underlying `httpx.Client` (which is *not* pickleable); the array
unpickles with a fresh transport.

In [ ]:
if HAS_DASK:
    import pickle
    ds = xr.open_dataset(collection_url, engine="edr", chunks={"t": 1})
    payload = pickle.dumps(ds["temperature"].variable.data)
    restored = pickle.loads(payload)
    print("pickle bytes:", len(payload))
    print("restored type:", type(restored).__name__)
    ds.close()
else:
    print("skipped — dask not installed")